# Script 2 – Finding and Replacing Wrong Data

## DEFINE WRONG DATA:
- **Soil Moisture (SWC_*):** Values must be between 0 and 0.6.
- **Precipitation (Ppt):** Should not be negative.
- **RH:** Must be between 0 and 100%.
- **Wind Speed:** Must be between 0 and 25 m/s.
- **Wind Direction:** Must be between 0° and 360°.
- **Solar Radiation (Srad):** Must be non-negative.
- **Air/Soil Temperature:** Can be limited to a plausible range (-30°C to 60°C).

The function finds the "wrong" values in each column, replaces them with NaN, and records the dates where replacements occurred.

In [1]:
import argparse
import pandas as pd
import numpy as np
from pathlib import Path
import os
import warnings
warnings.filterwarnings("ignore")

In [23]:
def load_merged_data(station_id, base_dir="raw_merged_data"):
    file_path = Path(base_dir) / f"raw_merged_station_{station_id}.csv"
    df = pd.read_csv(file_path, index_col=0, parse_dates=True)
    return df

In [3]:
def find_missing_data(df):
    missing_info = {}
    for col in df.columns:
        missing_dates = df.index[df[col].isnull()].tolist()
        if missing_dates:
            missing_info[col] = missing_dates
    return missing_info

In [4]:
def find_and_replace_wrong_data(df):
    wrong_info = {}
    df_corrected = df.copy()
    
    # Soil Moisture: between 0 and 0.6
    swc_cols = ['SWC_5', 'SWC_10', 'SWC_20', 'SWC_50']
    for col in swc_cols:
        if col in df_corrected.columns:
            wrong_idx = df_corrected.index[(df_corrected[col] < 0) | (df_corrected[col] > 0.6)].tolist()
            if wrong_idx:
                wrong_info[col] = wrong_idx
                df_corrected.loc[(df_corrected[col] < 0) | (df_corrected[col] > 0.6), col] = np.nan

    # Precipitation: non-negative
    if 'Ppt' in df_corrected.columns:
        wrong_idx = df_corrected.index[df_corrected['Ppt'] < 0].tolist()
        if wrong_idx:
            wrong_info['Ppt'] = wrong_idx
            df_corrected.loc[df_corrected['Ppt'] < 0, 'Ppt'] = np.nan

    # RH: between 0 and 100%
    if 'RH' in df_corrected.columns:
        wrong_idx = df_corrected.index[(df_corrected['RH'] < 0) | (df_corrected['RH'] > 100)].tolist()
        if wrong_idx:
            wrong_info['RH'] = wrong_idx
            df_corrected.loc[(df_corrected['RH'] < 0) | (df_corrected['RH'] > 100), 'RH'] = np.nan

    # Wind speed: 0 to 25 m/s
    if 'Wind speed' in df_corrected.columns:
        wrong_idx = df_corrected.index[(df_corrected['Wind speed'] < 0) | (df_corrected['Wind speed'] > 25)].tolist()
        if wrong_idx:
            wrong_info['Wind speed'] = wrong_idx
            df_corrected.loc[(df_corrected['Wind speed'] < 0) | (df_corrected['Wind speed'] > 25), 'Wind speed'] = np.nan

    # Wind direction: 0 to 360°
    if 'Wind direction' in df_corrected.columns:
        wrong_idx = df_corrected.index[(df_corrected['Wind direction'] < 0) | (df_corrected['Wind direction'] > 360)].tolist()
        if wrong_idx:
            wrong_info['Wind direction'] = wrong_idx
            df_corrected.loc[(df_corrected['Wind direction'] < 0) | (df_corrected['Wind direction'] > 360), 'Wind direction'] = np.nan

    # Solar radiation: must be non-negative
    if 'Srad' in df_corrected.columns:
        wrong_idx = df_corrected.index[df_corrected['Srad'] < 0].tolist()
        if wrong_idx:
            wrong_info['Srad'] = wrong_idx
            df_corrected.loc[df_corrected['Srad'] < 0, 'Srad'] = np.nan

    # Soil Temperature: -30°C to 60°C
    temp_cols = ['T_5', 'T_10', 'T_20', 'T_50']
    for col in temp_cols:
        if col in df_corrected.columns:
            wrong_idx = df_corrected.index[(df_corrected[col] < -30) | (df_corrected[col] > 60)].tolist()
            if wrong_idx:
                wrong_info[col] = wrong_info.get(col, []) + wrong_idx
                df_corrected.loc[(df_corrected[col] < -30) | (df_corrected[col] > 60), col] = np.nan

    # Air Temperature: -30°C to 60°C
    if 'Tair' in df_corrected.columns:
        wrong_idx = df_corrected.index[(df_corrected['Tair'] < -30) | (df_corrected['Tair'] > 60)].tolist()
        if wrong_idx:
            wrong_info['Tair'] = wrong_idx
            df_corrected.loc[(df_corrected['Tair'] < -30) | (df_corrected['Tair'] > 60), 'Tair'] = np.nan

    
    return df_corrected, wrong_info

In [5]:
def combine_nan_lists(missing_info, wrong_info):
    combined = {}
    all_cols = set(missing_info.keys()).union(wrong_info.keys())
    for col in all_cols:
        combined[col] = sorted(missing_info.get(col, []) + wrong_info.get(col, []))
    return combined

In [6]:
def detailed_missing_summary(missing_info):
    for col, dates in missing_info.items():
        print(f"\nColumn: {col}")
        total_missing = len(dates)
        print(f"  Total missing timestamps: {total_missing}")
        if total_missing == 0:
            continue
        
        # Convert dates list into a pandas Series for easy grouping
        date_series = pd.Series(dates)
        df_dates = pd.DataFrame({'Date': date_series})
        df_dates['Year'] = df_dates['Date'].dt.year
        df_dates['Month'] = df_dates['Date'].dt.month
        
        # Group by Year and Month
        grouped = df_dates.groupby(['Year', 'Month'])
        
        for (year, month), group in grouped:
            count = group.shape[0]
            first_ts = group['Date'].min()
            last_ts = group['Date'].max()
            print(f"  {year}-{month:02d}: Count = {count}, First = {first_ts}, Last = {last_ts}")



In [7]:

#def analyze_missing_patterns(missing_info):
    """Fixed version working with proper pandas structures"""
    results = {}
    
    for col, dates in missing_info.items():
        if not dates:
            continue
            
        # Convert to DataFrame
        df = pd.DataFrame({'dt': pd.to_datetime(dates)})
        df = df.sort_values('dt')
        
        # Detect continuous gaps
        df['gap_group'] = (df['dt'].diff().dt.total_seconds() > 3600).cumsum()
        
        # Aggregate using pandas-native syntax
        gaps = df.groupby('gap_group').agg(
            start=('dt', 'min'),
            end=('dt', 'max'),
            count=('dt', 'size')
        ).reset_index(drop=True)
        
        # Classify gaps
        results[col] = []
        for _, row in gaps.iterrows():
            duration = (row['end'] - row['start']).total_seconds()/3600
            results[col].append({
                'start': row['start'],
                'end': row['end'],
                'duration_hrs': duration,
                'classification': 'Short' if duration <=24 else 'Medium' if duration<=336 else 'Long',
                'count': row['count']
            })
    
    return results

#def print_analysis(results):
    """Clean formatted output"""
    for col, gaps in results.items():
        print(f"\n=== {col} ===")
        print(f"Total missing points: {sum(g['count'] for g in gaps)}")
        
        for gap in gaps:
            print(f"\n• {gap['start']} to {gap['end']}")
            print(f"  Duration: {gap['duration_hrs']:.1f} hrs ({gap['classification']})")
            print(f"  Missing points: {gap['count']}")



In [8]:
def group_consecutive_dates(dates, freq=pd.Timedelta(hours=1)):
    """
    Group sorted timestamps into consecutive blocks based on the expected frequency.
    
    Args:
        dates (list of pd.Timestamp): Sorted list of timestamps.
        freq (pd.Timedelta): Expected frequency (default is 1 hour).
        
    Returns:
        list of lists: Each inner list contains a group of consecutive timestamps.
    """
    groups = []
    if not dates:
        return groups
    current_group = [dates[0]]
    for prev, curr in zip(dates, dates[1:]):
        # If the time difference is less than or equal to 1.5 times the expected frequency,
        # consider the dates as consecutive.
        if curr - prev <= freq * 1.5:
            current_group.append(curr)
        else:
            groups.append(current_group)
            current_group = [curr]
    groups.append(current_group)
    return groups

In [9]:
def create_missing_summary_df(missing_info):
    """
    Create a summary DataFrame from missing data information.
    
    The summary DataFrame includes columns:
        Parameter, Start Timestamp, End Timestamp, Number Missing
    
    Args:
        missing_info (dict): Dictionary mapping each column to a list of missing timestamps.
    
    Returns:
        pd.DataFrame: Summary DataFrame.
    """
    summary_rows = []
    for param, dates in missing_info.items():
        sorted_dates = sorted(dates)
        groups = group_consecutive_dates(sorted_dates, freq=pd.Timedelta(hours=1))
        for group in groups:
            start_ts = group[0]
            end_ts = group[-1]
            count = len(group)
            summary_rows.append({
                "Parameter": param,
                "Start Timestamp": start_ts,
                "End Timestamp": end_ts,
                "Number Missing": count
            })
    summary_df = pd.DataFrame(summary_rows)
    return summary_df

1. Soil Moisture (SWC_5, SWC_10, SWC_20, SWC_50) and Soil Temperature (T_5, T_10, T_20, T_50)：
- short (several hours,  <24 hours): linear interpolation or forward/backward fill
- middle (1 day to 2 weeks): average values from the previous and next day
- long (>2 weeks): seasonal ARIMA,ort-term.

Method: Spline interpolation for Tair/RH; persistence for wind.